# 1. Method Choice and Why

I selected K-Means clustering because the FlyRank content-performance problem does not have a predefined target label.

The goal is to identify natural groups of content based on search visibility, website traffic, and user engagement signals.

I used 14 numerical features, including impressions, clicks, search position, pageviews, sessions, users, traffic sources, engagement metrics, and scroll events.

K-Means is appropriate because it groups similar content items without requiring predefined Higher-Performance or Lower-Performance labels.

The baseline analysis provides a comparison point, while the clustering model is evaluated using silhouette scores and interpretation of the resulting content-performance archetypes.

In [2]:
!pip install -q duckdb pandas numpy scikit-learn pyarrow

In [ ]:
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = "PASTE_YOUR_HUGGINGFACE_READ_TOKEN_HERE"

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN 
)
""")

dataset = "hf://datasets/FlyRank/internship-warehouse"

print("Connected to FlyRank Internship Warehouse")

Connected to FlyRank Internship Warehouse


In [4]:
dataset = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{dataset}/fact_content_daily_performance_sample.parquet'
    )
    LIMIT 50000
""").df()

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(50000, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [12]:
df = con.sql(f"""
SELECT
    content_hash_id,

    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    AVG(gsc_sum_position) AS avg_position,

    SUM(ga4_pageviews) AS total_pageviews,
    SUM(sessions_organic + sessions_direct + sessions_referral + sessions_social + sessions_ai + sessions_paid) AS total_sessions,
    SUM(ga4_users) AS total_users,
    SUM(ga4_engaged_sessions) AS total_engaged_sessions,
    SUM(ga4_total_engagement_sec) AS total_engagement_time,

    SUM(sessions_organic) AS organic_sessions,
    SUM(sessions_direct) AS direct_sessions,
    SUM(sessions_referral) AS referral_sessions,
    SUM(sessions_social) AS social_sessions,
    SUM(sessions_ai) AS ai_sessions,
    SUM(sessions_paid) AS paid_sessions,

    SUM(scroll_events) AS total_scroll_events

FROM read_parquet(
    '{dataset}/fact_content_daily_performance/**/*.parquet'
)

GROUP BY content_hash_id
LIMIT 50000
""").df()

print("Content-level dataset shape:", df.shape)

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content-level dataset shape: (50000, 16)


,content_hash_id,total_impressions,total_clicks,avg_position,total_pageviews,total_sessions,total_users,total_engaged_sessions,total_engagement_time,organic_sessions,direct_sessions,referral_sessions,social_sessions,ai_sessions,paid_sessions,total_scroll_events
0,content_3b70a18ea133b2bb,12277.0,51.0,358.869231,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,content_c899aef92518c714,30756.0,115.0,710.953846,56.0,54.0,47.0,5.0,740.0,42.0,5.0,2.0,0.0,5.0,0.0,30.0
2,content_c7c1d2e68d9d0964,6016.0,18.0,219.053846,15.0,6.0,15.0,0.0,0.0,4.0,0.0,2.0,0.0,0.0,0.0,2.0
3,content_ae5e5fd6edff550f,14482.0,26.0,239.265385,33.0,3.0,31.0,2.0,170.0,0.0,3.0,0.0,0.0,0.0,0.0,8.0
4,content_a64143f6e4a21ffe,411801.0,2947.0,3351.457692,444.0,528.0,418.0,34.0,4849.0,502.0,25.0,1.0,0.0,0.0,0.0,209.0


In [13]:
print("Missing values before cleaning:")
print(df.isnull().sum())

df["avg_position"] = df["avg_position"].fillna(
    df["avg_position"].median()
)

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns

df[numeric_columns] = df[numeric_columns].fillna(0)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
content_hash_id           0
total_impressions         0
total_clicks              0
avg_position              0
total_pageviews           0
total_sessions            0
total_users               0
total_engaged_sessions    0
total_engagement_time     0
organic_sessions          0
direct_sessions           0
referral_sessions         0
social_sessions           0
ai_sessions               0
paid_sessions             0
total_scroll_events       0
dtype: int64

Missing values after cleaning:
content_hash_id           0
total_impressions         0
total_clicks              0
avg_position              0
total_pageviews           0
total_sessions            0
total_users               0
total_engaged_sessions    0
total_engagement_time     0
organic_sessions          0
direct_sessions           0
referral_sessions         0
social_sessions           0
ai_sessions               0
paid_sessions             0
total_scroll_events       0
dtype: int64


In [14]:
features = [
    "total_impressions",
    "total_clicks",
    "avg_position",
    "total_pageviews",
    "total_sessions",
    "total_users",
    "total_engaged_sessions",
    "total_engagement_time",
    "organic_sessions",
    "direct_sessions",
    "referral_sessions",
    "social_sessions",
    "ai_sessions",
    "total_scroll_events"
]

X = df[features].copy()

print("Number of features:", len(features))
print("Dataset shape:", X.shape)

Number of features: 14
Dataset shape: (50000, 14)


## 2. Split Design

K-Means is an unsupervised learning method, so there is no target label to stratify.

The content items are split into 80% training data and 20% held-out evaluation data using a fixed random seed.

The scaler is fitted only on the training data. The same fitted scaler is then used to transform the evaluation data. This avoids using information from the evaluation set during preprocessing.

The K-Means model is fitted on the training data and evaluated on the held-out data using the silhouette score.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test = train_test_split(
    X,
    test_size=0.20,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train.shape)
print("Evaluation shape:", X_test.shape)

Training shape: (40000, 14)
Evaluation shape: (10000, 14)


In [16]:
from sklearn.metrics import silhouette_score

baseline_result = "One-cluster baseline"

print("Baseline:", baseline_result)
print("Silhouette score cannot be calculated for one cluster.")

Baseline: One-cluster baseline
Silhouette score cannot be calculated for one cluster.


In [20]:
from sklearn.cluster import KMeans

k_values = range(2, 11)
results = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_train_scaled)

    train_silhouette = silhouette_score(X_train_scaled, kmeans.labels_)
    test_silhouette = silhouette_score(X_test_scaled, kmeans.predict(X_test_scaled))

    results.append({
        "k": k,
        "train_silhouette": train_silhouette,
        "test_silhouette": test_silhouette
    })

results_df = pd.DataFrame(results)

best_row = results_df.loc[
    results_df["test_silhouette"].idxmax()
]

best_k = int(best_row["k"])

print("Best number of clusters:", best_k)
print("Best evaluation silhouette score:",
      best_row["test_silhouette"])

Best number of clusters: 2
Best evaluation silhouette score: 0.9237015104082613


In [21]:
final_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

final_train_labels = final_model.fit_predict(
    X_train_scaled
)

final_test_labels = final_model.predict(
    X_test_scaled
)

print("Final model created successfully")

Final model created successfully


In [22]:
comparison_table = pd.DataFrame({
    "Approach": [
        "Week 4 Baseline",
        f"Week 5 K-Means (K={best_k})"
    ],
    "Method": [
        "Single-group baseline",
        "Unsupervised K-Means clustering"
    ],
    "Evaluation Metric": [
        "Not applicable for one cluster",
        "Held-out silhouette score"
    ],
    "Result": [
        "N/A",
        round(float(best_row["test_silhouette"]), 4)
    ]
})

comparison_table

,Approach,Method,Evaluation Metric,Result
0,Week 4 Baseline,Single-group baseline,Not applicable for one cluster,N/A
1,Week 5 K-Means (K=2),Unsupervised K-Means clustering,Held-out silhouette score,0.9237


In [23]:
test_results = X_test.copy()

test_results["cluster"] = final_test_labels

cluster_summary = test_results.groupby(
    "cluster"
)[features].mean()

cluster_summary

,total_impressions,total_clicks,avg_position,total_pageviews,total_sessions,total_users,total_engaged_sessions,total_engagement_time,organic_sessions,direct_sessions,referral_sessions,social_sessions,ai_sessions,total_scroll_events
cluster,,,,,,,,,,,,,,
0,11379.139348,34.698772,306.837265,26.623059,16.171014,21.318786,0.643357,87.97686,9.846443,3.271998,0.209073,0.016645,0.113164,2.899016
1,179876.823129,948.979592,4031.153929,1304.598639,1063.659864,763.156463,39.265306,5359.44898,793.809524,53.700680,9.163265,1.598639,4.857143,73.149660


In [24]:
key_metrics = [
    "total_impressions",
    "total_clicks",
    "total_pageviews",
    "organic_sessions",
    "total_scroll_events"
]

normalized_summary = (
    cluster_summary[key_metrics] -
    cluster_summary[key_metrics].min()
) / (
    cluster_summary[key_metrics].max() -
    cluster_summary[key_metrics].min()
)

cluster_performance_score = normalized_summary.mean(axis=1)

cluster_ranking = pd.DataFrame({
    "performance_score": cluster_performance_score
}).sort_values(
    "performance_score",
    ascending=False
)

cluster_ranking

,performance_score
cluster,
1,1.0
0,0.0


In [25]:
best_cluster = cluster_ranking.index[0]
lowest_cluster = cluster_ranking.index[-1]

archetype_map = {}

for cluster in cluster_ranking.index:
    if cluster == best_cluster:
        archetype_map[cluster] = "Higher-Performance Content"
    elif cluster == lowest_cluster:
        archetype_map[cluster] = "Lower-Performance Content"
    else:
        archetype_map[cluster] = f"Intermediate-Performance Content {cluster}"

test_results["content_archetype"] = test_results[
    "cluster"
].map(archetype_map)

print(test_results["content_archetype"].value_counts())

content_archetype
Lower-Performance Content     9853
Higher-Performance Content     147
Name: count, dtype: int64


## 4. Errors and Interpretation

This is an unsupervised learning problem, so there are no ground-truth class labels and therefore no conventional classification errors such as false positives or false negatives.

Model quality was evaluated using the silhouette score on held-out content items.

Potential limitations include:

1. Content near the boundary between clusters may be assigned differently with another sample or clustering method.

2. K-Means is sensitive to feature scaling, which is why StandardScaler was applied using training data only.

3. Cluster numbers do not have an inherent meaning. The clusters were interpreted after modeling using their observed performance characteristics.

4. The model identifies associations and patterns. It does not establish that impressions, clicks, engagement, or any other feature causes content success.

The model should therefore be used for analysis and prioritization rather than automatic business decisions.

In [26]:
print("WEEK 05 MODEL - SELF CHECK")

print("\n1. Method choice:")
print("K-Means clustering selected for an unlabeled content-performance problem.")

print("\n2. Split design:")
print(f"Training items: {len(X_train)}")
print(f"Evaluation items: {len(X_test)}")

print("\n3. Model comparison:")
print(comparison_table.to_string(index=False))

print("\n4. Selected model:")
print(f"Best K: {best_k}")
print(f"Held-out silhouette score: {best_row['test_silhouette']:.4f}")

print("\n5. Interpretation:")
print(test_results["content_archetype"].value_counts())

print("\n6. Limit:")
print("Results describe observed patterns and do not prove causation.")

print("\nWEEK 05 SELF-CHECK COMPLETE")

WEEK 05 MODEL - SELF CHECK

1. Method choice:
K-Means clustering selected for an unlabeled content-performance problem.

2. Split design:
Training items: 40000
Evaluation items: 10000

3. Model comparison:
            Approach                          Method              Evaluation Metric  Result
     Week 4 Baseline           Single-group baseline Not applicable for one cluster     N/A
Week 5 K-Means (K=2) Unsupervised K-Means clustering      Held-out silhouette score  0.9237

4. Selected model:
Best K: 2
Held-out silhouette score: 0.9237

5. Interpretation:
content_archetype
Lower-Performance Content     9853
Higher-Performance Content     147
Name: count, dtype: int64

6. Limit:
Results describe observed patterns and do not prove causation.

WEEK 05 SELF-CHECK COMPLETE
